# Time to slice and dice

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
# Install required libraries.
# Run once before executing any other cells.

!pip install datasets evaluate transformers[sentencepiece]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00


In [2]:
# Download the Drug Review dataset as a .zip file and extract it.
# 'wget' downloads the file; 'unzip' extracts it.
# We'll get two .tsv files: drugsComTrain_raw.tsv and drugsComTest_raw.tsv.

!wget "https://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip"
!unzip drugsCom_raw.zip

--2026-03-24 13:55:00--  https://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘drugsCom_raw.zip’

drugsCom_raw.zip        [         <=>        ]  41.00M  13.1MB/s    in 3.1s    

2026-03-24 13:55:03 (13.1 MB/s) - ‘drugsCom_raw.zip’ saved [42989872]

Archive:  drugsCom_raw.zip
  inflating: drugsComTest_raw.tsv    
  inflating: drugsComTrain_raw.tsv   


In [3]:
# Load the TSV (Tab-Separated Values) files using the 'csv' loader.
# TSV is the same as CSV but uses tabs instead of commas.
# 'delimiter="\t"' tells the loader to split columns on tab characters.

from datasets import load_dataset

data_files = {"train": "drugsComTrain_raw.tsv", "test": "drugsComTest_raw.tsv"}
# \t is the tab character in Python
drug_dataset = load_dataset("csv", data_files=data_files, delimiter="\t")

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [4]:
# Take a random sample of 1000 examples from the training set for quick inspection.
# .shuffle(seed=42) randomly reorders (seed=42 makes it reproducible).
# .select(range(1000)) picks the first 1000 from the shuffled result.
# This gives us a quick feel for the data without loading everything.

drug_sample = drug_dataset["train"].shuffle(seed=42).select(range(1000))
# Print the first few examples of the dataset
drug_sample[:3]

{'Unnamed: 0': [87571, 178045, 80482],
 'drugName': ['Naproxen', 'Duloxetine', 'Mobic'],
 'condition': ['Gout, Acute', 'ibromyalgia', 'Inflammatory Conditions'],
 'review': ['"like the previous person mention, I&#039;m a strong believer of aleve, it works faster for my gout than the prescription meds I take. No more going to the doctor for refills.....Aleve works!"',
  '"I have taken Cymbalta for about a year and a half for fibromyalgia pain. It is great\r\nas a pain reducer and an anti-depressant, however, the side effects outweighed \r\nany benefit I got from it. I had trouble with restlessness, being tired constantly,\r\ndizziness, dry mouth, numbness and tingling in my feet, and horrible sweating. I am\r\nbeing weaned off of it now. Went from 60 mg to 30mg and now to 15 mg. I will be\r\noff completely in about a week. The fibro pain is coming back, but I would rather deal with it than the side effects."',
  '"I have been taking Mobic for over a year with no side effects other than 

In [5]:
# Verify that 'Unnamed: 0' is a unique patient ID column.
# .unique() returns all distinct values; its length should equal the total row count.
# 'assert' will raise an error if the condition is False — a quick sanity check.

for split in drug_dataset.keys():
    assert len(drug_dataset[split]) == len(drug_dataset[split].unique("Unnamed: 0"))

In [6]:
# Rename the cryptic 'Unnamed: 0' column to the more meaningful 'patient_id'.
# .rename_column() works across ALL splits in the DatasetDict at once.
# Both train and test splits are updated in a single call.

drug_dataset = drug_dataset.rename_column(
    original_column_name="Unnamed: 0", new_column_name="patient_id"
)
drug_dataset

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 161297
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 53766
    })
})

In [7]:
# Attempt to lowercase all condition labels.
# This WILL fail because some 'condition' values are None (missing data).
# Calling .lower() on None raises AttributeError — shown in the output below.

def lowercase_condition(example):
    return {"condition": example["condition"].lower()}


drug_dataset.map(lowercase_condition)

Map:   0%|          | 0/161297 [00:00<?, ? examples/s]

AttributeError: 'NoneType' object has no attribute 'lower'

In [8]:
# Define a regular named filter function for comparison.
# This is the explicit (non-lambda) way to write a filter:
# return True to keep the row, False to drop it.

def filter_nones(x):
    return x["condition"] is not None

In [9]:
# Quick demo: lambda functions can square a number.
# 'lambda x: x * x' is a compact function that takes x and returns x*x.
# (lambda ...)(3) immediately calls it with argument 3 → result is 9.

(lambda x: x * x)(3)

9

In [10]:
# Another lambda demo: compute triangle area.
# 'lambda base, height: 0.5 * base * height' takes two args.
# Calling with (4, 8) → 0.5 × 4 × 8 = 16.0

(lambda base, height: 0.5 * base * height)(4, 8)

16.0

In [11]:
# Filter out rows where 'condition' is None using a lambda.
# This is the compact, one-line alternative to the filter_nones function above.
# After this, every row is guaranteed to have a non-null condition.

drug_dataset = drug_dataset.filter(lambda x: x["condition"] is not None)

Filter:   0%|          | 0/161297 [00:00<?, ? examples/s]

Filter:   0%|          | 0/53766 [00:00<?, ? examples/s]

In [12]:
# Now lowercase the condition column safely (no more None values).
# .map() applies the function to every row.
# The first 3 results verify it worked correctly.

drug_dataset = drug_dataset.map(lowercase_condition)
# Check that lowercasing worked
drug_dataset["train"]["condition"][:3]

Map:   0%|          | 0/160398 [00:00<?, ? examples/s]

Map:   0%|          | 0/53471 [00:00<?, ? examples/s]

['left ventricular dysfunction', 'adhd', 'birth control']

In [13]:
# Define a function to count words in each review.
# .split() splits on whitespace, len() counts the words.
# Returning a NEW key 'review_length' adds it as a brand new column.

def compute_review_length(example):
    return {"review_length": len(example["review"].split())}

In [14]:
# Apply the word-count function to every row in the dataset.
# This creates a new 'review_length' column in all splits.
# Check the first training example — it now shows 'review_length': 17.

drug_dataset = drug_dataset.map(compute_review_length)
# Inspect the first training example
drug_dataset["train"][0]

Map:   0%|          | 0/160398 [00:00<?, ? examples/s]

Map:   0%|          | 0/53471 [00:00<?, ? examples/s]

{'patient_id': 206461,
 'drugName': 'Valsartan',
 'condition': 'left ventricular dysfunction',
 'review': '"It has no side effect, I take it in combination of Bystolic 5 Mg and Fish Oil"',
 'rating': 9.0,
 'date': 'May 20, 2012',
 'usefulCount': 27,
 'review_length': 17}

In [15]:
# Sort training examples by review_length to find the shortest ones.
# The shortest reviews ('Excellent.', 'useless', 'ok') have review_length = 1.
# These single-word reviews are not useful for training models.

drug_dataset["train"].sort("review_length")[:3]

{'patient_id': [111469, 13653, 53602],
 'drugName': ['Ledipasvir / sofosbuvir',
  'Amphetamine / dextroamphetamine',
  'Alesse'],
 'condition': ['hepatitis c', 'adhd', 'birth control'],
 'review': ['"Headache"', '"Great"', '"Awesome"'],
 'rating': [10.0, 10.0, 10.0],
 'date': ['February 3, 2015', 'October 20, 2009', 'November 23, 2015'],
 'usefulCount': [41, 3, 0],
 'review_length': [1, 1, 1]}

In [16]:
# Filter out very short reviews (fewer than 30 words).
# This removes ~15% of reviews that are too brief to be informative.
# .num_rows shows how many examples remain after filtering.

drug_dataset = drug_dataset.filter(lambda x: x["review_length"] > 30)
print(drug_dataset.num_rows)

Filter:   0%|          | 0/160398 [00:00<?, ? examples/s]

Filter:   0%|          | 0/53471 [00:00<?, ? examples/s]

{'train': 138514, 'test': 46108}


In [17]:
# Demo: html.unescape() converts HTML entities to real characters.
# '&#039;' → apostrophe ('), '&amp;' → '&', etc.
# The drug reviews contain these HTML codes that need to be cleaned.

import html

text = "I&#039;m a transformer called BERT"
html.unescape(text)

"I'm a transformer called BERT"

In [18]:
# Clean all HTML entities from reviews using .map().
# This fixes characters like &#039; → ' in every review row.

drug_dataset = drug_dataset.map(lambda x: {"review": html.unescape(x["review"])})

Map:   0%|          | 0/138514 [00:00<?, ? examples/s]

Map:   0%|          | 0/46108 [00:00<?, ? examples/s]

In [19]:
# Same HTML cleaning but using batched=True for much faster processing.
# batched=True passes a batch of 1000 rows at once instead of one-by-one.
# List comprehension [html.unescape(o) for o in x["review"]] processes the whole batch at once.

new_drug_dataset = drug_dataset.map(
    lambda x: {"review": [html.unescape(o) for o in x["review"]]}, batched=True
)

Map:   0%|          | 0/138514 [00:00<?, ? examples/s]

Map:   0%|          | 0/46108 [00:00<?, ? examples/s]

In [20]:
# Load a fast BERT tokenizer and define a tokenize function.
# AutoTokenizer auto-selects the right tokenizer for 'bert-base-cased'.
# 'truncation=True' cuts reviews that are too long for the model.

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")


def tokenize_function(examples):
    return tokenizer(examples["review"], truncation=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [21]:
# Tokenize the entire dataset with batched=True.
# %time measures how long this takes.
# batched=True with a fast tokenizer is ~30x faster than without batching.

%time tokenized_dataset = drug_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/138514 [00:00<?, ? examples/s]

Map:   0%|          | 0/46108 [00:00<?, ? examples/s]

CPU times: user 1min 56s, sys: 566 ms, total: 1min 56s
Wall time: 1min 19s


In [22]:
# Compare fast vs slow tokenizer performance.
# 'use_fast=False' forces the slower Python-based tokenizer.
# 'num_proc=8' uses 8 CPU processes in parallel to compensate for the slowness.

slow_tokenizer = AutoTokenizer.from_pretrained("bert-base-cased", use_fast=False)


def slow_tokenize_function(examples):
    return slow_tokenizer(examples["review"], truncation=True)


tokenized_dataset = drug_dataset.map(slow_tokenize_function, batched=True, num_proc=8)

Map (num_proc=8):   0%|          | 0/138514 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/46108 [00:00<?, ? examples/s]

In [23]:
# Define tokenize_and_split() which splits long reviews into chunks of max 128 tokens.
# 'return_overflowing_tokens=True' returns ALL chunks, not just the first one.
# A review of 200 tokens → [128 tokens, 72 tokens] = 2 examples.

def tokenize_and_split(examples):
    return tokenizer(
        examples["review"],
        truncation=True,
        max_length=128,
        return_overflowing_tokens=True,
    )

In [24]:
# Test tokenize_and_split on one example.
# [128, 49] means the review was split into 2 chunks: 128 tokens and 49 tokens.
# This is expected behaviour when a review is longer than max_length=128.

result = tokenize_and_split(drug_dataset["train"][0])

[len(inp) for inp in result["input_ids"]]

[128, 49]

In [25]:
# This will FAIL because column length mismatches occur.
# 1000 input rows → 1463 output chunks (some reviews get split into 2+).
# The existing columns still have 1000 values but tokenized output has 1463 → shape error.

tokenized_dataset = drug_dataset.map(tokenize_and_split, batched=True)

Map:   0%|          | 0/138514 [00:00<?, ? examples/s]

ArrowInvalid: Column 8 named input_ids expected length 1000 but got length 1463

In [26]:
# Fix the length mismatch by removing old columns.
# 'remove_columns=drug_dataset["train"].column_names' drops all original columns.
# Now only the tokenizer output columns remain — no length conflict.

tokenized_dataset = drug_dataset.map(
    tokenize_and_split, batched=True, remove_columns=drug_dataset["train"].column_names
)

Map:   0%|          | 0/138514 [00:00<?, ? examples/s]

Map:   0%|          | 0/46108 [00:00<?, ? examples/s]

In [27]:
# Verify: tokenized dataset has MORE rows than original.
# 206772 chunks from 138514 original reviews — because long reviews were split.

len(tokenized_dataset["train"]), len(drug_dataset["train"])

(206772, 138514)

In [28]:
# Alternative fix: keep ALL original columns by mapping their values.
# 'overflow_to_sample_mapping' tells us which original row each chunk came from.
# We use it to replicate the original column values for each chunk.

def tokenize_and_split(examples):
    result = tokenizer(
        examples["review"],
        truncation=True,
        max_length=128,
        return_overflowing_tokens=True,
    )
    # Extract mapping between new and old indices
    sample_map = result.pop("overflow_to_sample_mapping")
    for key, values in examples.items():
        result[key] = [values[i] for i in sample_map]
    return result

In [29]:
# Apply the improved tokenize_and_split — now keeping all original columns.
# The dataset grows from 138514 to 206772 rows but retains all metadata columns.

tokenized_dataset = drug_dataset.map(tokenize_and_split, batched=True)
tokenized_dataset

Map:   0%|          | 0/138514 [00:00<?, ? examples/s]

Map:   0%|          | 0/46108 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 206772
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 68876
    })
})

In [30]:
# Switch the dataset format to Pandas for analysis.
# .set_format('pandas') changes the output type of __getitem__.
# Accessing dataset elements now returns pd.DataFrame instead of dict.

drug_dataset.set_format("pandas")

In [31]:
# Preview the first 3 rows as a Pandas DataFrame.
# The output is a nicely formatted table instead of a Python dict.

drug_dataset["train"][:3]

,patient_id,drugName,condition,review,rating,date,usefulCount,review_length
0,95260,Guanfacine,adhd,"""My son is halfway through his fourth week of ...",8.0,"April 27, 2010",192,141
1,92703,Lybrel,birth control,"""I used to take another oral contraceptive, wh...",5.0,"December 14, 2009",17,134
2,138000,Ortho Evra,birth control,"""This is my first time using any form of birth...",8.0,"November 3, 2015",10,89


In [32]:
# Load ALL training data into a Pandas DataFrame.
# '[:]' slices the entire dataset.
# Note: drug_dataset['train'] is still a Dataset object — the slice creates the DataFrame.

train_df = drug_dataset["train"][:]

In [33]:
# Use Pandas to compute how many reviews exist per condition.
# .value_counts() counts occurrences of each unique condition.
# .to_frame(), .reset_index(), .rename() tidy up the resulting DataFrame.

frequencies = (
    train_df["condition"]
    .value_counts()
    .to_frame()
    .reset_index()
    .rename(columns={"index": "condition", "condition": "frequency"})
)
frequencies.head()

,frequency,count
0,birth control,27655
1,depression,8023
2,acne,5209
3,anxiety,4991
4,pain,4744


In [34]:
# Convert the Pandas DataFrame back into a HuggingFace Dataset.
# Dataset.from_pandas() wraps any DataFrame as a Dataset.
# Now we can use all HuggingFace Dataset methods on this frequency table.

from datasets import Dataset

freq_dataset = Dataset.from_pandas(frequencies)
freq_dataset

Dataset({
    features: ['frequency', 'count'],
    num_rows: 819
})

In [35]:
# Reset the output format back to the default (Apache Arrow / dict).
# Always do this after using set_format('pandas') if you want to continue
# using .map(), .filter(), etc. (they don't work in pandas mode).

drug_dataset.reset_format()

In [36]:
# Split the training data into train (80%) and validation (20%) sets.
# train_test_split() is similar to scikit-learn's version.
# .pop('test') renames the auto-created 'test' split to 'validation',
# then we add the real test set separately.

drug_dataset_clean = drug_dataset["train"].train_test_split(train_size=0.8, seed=42)
# Rename the default "test" split to "validation"
drug_dataset_clean["validation"] = drug_dataset_clean.pop("test")
# Add the "test" set to our `DatasetDict`
drug_dataset_clean["test"] = drug_dataset["test"]
drug_dataset_clean

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 110811
    })
    validation: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 27703
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 46108
    })
})

In [37]:
# Save the cleaned dataset to disk in Apache Arrow format.
# .save_to_disk() creates a folder with one sub-folder per split.
# Arrow is fast for reading back — ideal for ML training pipelines.

drug_dataset_clean.save_to_disk("drug-reviews")

Saving the dataset (0/1 shards):   0%|          | 0/110811 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/27703 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/46108 [00:00<?, ? examples/s]

In [38]:
# Load the saved Arrow dataset back from disk.
# load_from_disk() reconstructs the full DatasetDict from the saved folder.

from datasets import load_from_disk

drug_dataset_reloaded = load_from_disk("drug-reviews")
drug_dataset_reloaded

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 110811
    })
    validation: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 27703
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 46108
    })
})

In [39]:
# Save each split as a separate JSON Lines (.jsonl) file.
# Each line in the file is one complete JSON object (one row).
# Useful for sharing data with tools outside HuggingFace.

for split, dataset in drug_dataset_clean.items():
    dataset.to_json(f"drug-reviews-{split}.jsonl")

Creating json from Arrow format:   0%|          | 0/111 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/28 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/47 [00:00<?, ?ba/s]

In [ ]:
# Preview the first line of the saved JSONL file.
# Each line is a complete JSON record — easy to parse line-by-line.

!head -n 1 drug-reviews-train.jsonl

{"patient_id":141780,"drugName":"Escitalopram","condition":"depression","review":"\"I seemed to experience the regular side effects of LEXAPRO, insomnia, low sex drive, sleepiness during the day. I am taking it at night because my doctor said if it made me tired to take it at night. I assumed it would and started out taking it at night. Strange dreams, some pleasant. I was diagnosed with fibromyalgia. Seems to be helping with the pain. Have had anxiety and depression in my family, and have tried quite a few other medications that haven't worked. Only have been on it for two weeks but feel more positive in my mind, want to accomplish more in my life. Hopefully the side effects will dwindle away, worth it to stick with it from hearing others responses. Great medication.\"","rating":9.0,"date":"May 29, 2011","usefulCount":10,"review_length":125}

In [40]:
# Reload the JSONL files back as a DatasetDict.
# This confirms the round-trip: save → reload works correctly.

data_files = {
    "train": "drug-reviews-train.jsonl",
    "validation": "drug-reviews-validation.jsonl",
    "test": "drug-reviews-test.jsonl",
}
drug_dataset_reloaded = load_dataset("json", data_files=data_files)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]